# Description

In this notebook, we benchmark SINDy algorithm on the set of previously generated SR benchmarks.

In [1]:
from __future__ import annotations

import csv
import time
from pathlib import Path

import h5py
import numpy as np
import sympy as sp

from pysindy.feature_library import PolynomialLibrary, CustomLibrary, ConcatLibrary
from pysindy.optimizers import STLSQ

from config.benchmark_config import DataCFG, SINDY


# ------------------------------------------------------------
# Data loading (IDENTICAL STRUCTURE)
# ------------------------------------------------------------

def _load_one_group(f: h5py.File, gname: str):
    g = f[gname]
    raw = g["sympy_str"][()]
    true_expr_str = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else str(raw)

    Xtr = g["train"]["X"][...].astype(np.float64, copy=False)
    ytr = g["train"]["y"][...].astype(np.float64, copy=False).reshape(-1)

    Xti = g["test_interp"]["X"][...].astype(np.float64, copy=False)
    yti = g["test_interp"]["y"][...].astype(np.float64, copy=False).reshape(-1)

    Xte = g["test_extrap"]["X"][...].astype(np.float64, copy=False)
    yte = g["test_extrap"]["y"][...].astype(np.float64, copy=False).reshape(-1)

    return true_expr_str, Xtr, ytr, Xti, yti, Xte, yte


def _mse(yhat: np.ndarray, y: np.ndarray) -> float:
    yhat = np.asarray(yhat, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    return float(np.mean((yhat - y) ** 2))


def _feature_names(n_features: int) -> list[str]:
    return [f"x{i+1}" for i in range(n_features)]


# ------------------------------------------------------------
# Custom library (numerically safe)
# ------------------------------------------------------------

def make_custom_library(cfg):

    def log_(x):
        return np.nan_to_num(np.log(np.abs(x) + cfg.log_eps))

    def sqrt_(x):
        if cfg.sqrt_abs:
            return np.nan_to_num(np.sqrt(np.abs(x)))
        return np.nan_to_num(np.sqrt(x))

    def square_(x):
        return x ** 2

    def div_(a, b):
        return np.nan_to_num(a / (b + cfg.div_eps))

    unary_map = {
        "log": (log_, lambda x: f"log({x})"),
        "sqrt": (sqrt_, lambda x: f"sqrt({x})"),
        "square": (square_, lambda x: f"({x})^2"),
    }

    funcs = []
    names = []

    for op in cfg.unary_ops:
        f, n = unary_map[op]
        funcs.append(f)
        names.append(n)

    if "/" in cfg.binary_ops:
        funcs.append(div_)
        names.append(lambda a, b: f"({a})/({b})")

    return CustomLibrary(
        library_functions=funcs,
        function_names=names,
        include_bias=False,
    )


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

def main():

    cfg = DataCFG()
    out_csv = Path(SINDY.results_path)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    with h5py.File(cfg.h5_path, "r") as f, out_csv.open("w", newline="") as out:

        w = csv.writer(out)
        w.writerow(
            [
                "group",
                "run",
                "seed",
                "n_train",
                "n_features",
                "train_mse",
                "test_interp_mse",
                "test_extrap_mse",
                "duration_s",
                "true_expr",
                "found_expr",
            ]
        )

        groups = sorted(f.keys())

        for gname in groups:

            true_expr_str, Xtr, ytr, Xti, yti, Xte, yte = _load_one_group(f, gname)

            n_features = Xtr.shape[1]
            feature_names = _feature_names(n_features)

            poly = PolynomialLibrary(
                degree=SINDY.poly_degree,
                include_interaction=SINDY.include_interaction,
                include_bias=SINDY.include_bias,
            )

            custom = make_custom_library(SINDY)
            library = ConcatLibrary([poly, custom])
            library.fit(Xtr)

            successful_runs = 0
            attempt = 0

            while successful_runs < SINDY.n_runs:

                seed = SINDY.base_seed + attempt
                attempt += 1
                np.random.seed(seed)

                theta_tr = np.asarray(library.transform(Xtr), dtype=np.float64)
                theta_ti = np.asarray(library.transform(Xti), dtype=np.float64)
                theta_te = np.asarray(library.transform(Xte), dtype=np.float64)

                theta_tr = np.clip(theta_tr, -SINDY.max_feature_abs, SINDY.max_feature_abs)
                theta_ti = np.clip(theta_ti, -SINDY.max_feature_abs, SINDY.max_feature_abs)
                theta_te = np.clip(theta_te, -SINDY.max_feature_abs, SINDY.max_feature_abs)

                optimizer = STLSQ(
                    threshold=SINDY.threshold,
                    alpha=SINDY.alpha,
                    max_iter=SINDY.max_iter,
                    normalize_columns=SINDY.normalize_columns,
                )

                t0 = time.perf_counter()
                optimizer.fit(theta_tr, ytr.reshape(-1, 1))
                dur = time.perf_counter() - t0

                coef = optimizer.coef_[0]
                intercept = float(getattr(optimizer, "intercept_", 0.0))

                yhat_tr = theta_tr @ coef + intercept
                yhat_ti = theta_ti @ coef + intercept
                yhat_te = theta_te @ coef + intercept

                train_mse = _mse(yhat_tr, ytr)
                test_interp_mse = _mse(yhat_ti, yti)
                test_extrap_mse = _mse(yhat_te, yte)

                if not (
                    np.isfinite(train_mse)
                    and np.isfinite(test_interp_mse)
                    and np.isfinite(test_extrap_mse)
                ):
                    print(f"[{gname}] seed={seed} FAILED → retry")
                    continue

                # build sympy expression
                expr = sp.Float(0.0)
                symbols = {n: sp.Symbol(n) for n in feature_names}
                feature_names_full = library.get_feature_names(input_features=feature_names)

                for c, fname in zip(coef, feature_names_full):
                    if abs(c) <= SINDY.coef_zero_tol:
                        continue
                    safe_name = fname.replace("^", "**")

                    # Fix implicit multiplication: "x1 x2" → "x1*x2"
                    safe_name = safe_name.replace(" ", "*")

                    expr += sp.Float(c) * sp.sympify(
                        safe_name,
                        locals=symbols,
                    )

                if abs(intercept) > SINDY.coef_zero_tol:
                    expr += sp.Float(intercept)

                found_expr_str = str(expr)

                w.writerow(
                    [
                        gname,
                        successful_runs,
                        seed,
                        int(Xtr.shape[0]),
                        n_features,
                        train_mse,
                        test_interp_mse,
                        test_extrap_mse,
                        dur,
                        true_expr_str,
                        found_expr_str,
                    ]
                )
                out.flush()

                print(
                    f"[{gname}] run={successful_runs} seed={seed} "
                    f"train={train_mse:.3e} interp={test_interp_mse:.3e} "
                    f"extrap={test_extrap_mse:.3e} dur={dur:.2f}s"
                )

                successful_runs += 1

    print(f"\nSaved: {out_csv}")


if __name__ == "__main__":
    main()

[expr_000_lin_uni] run=0 seed=0 train=0.000e+00 interp=0.000e+00 extrap=0.000e+00 dur=0.00s
[expr_000_lin_uni] run=1 seed=1 train=0.000e+00 interp=0.000e+00 extrap=0.000e+00 dur=0.00s
[expr_000_lin_uni] run=2 seed=2 train=0.000e+00 interp=0.000e+00 extrap=0.000e+00 dur=0.00s
[expr_000_lin_uni] run=3 seed=3 train=0.000e+00 interp=0.000e+00 extrap=0.000e+00 dur=0.00s
[expr_000_lin_uni] run=4 seed=4 train=0.000e+00 interp=0.000e+00 extrap=0.000e+00 dur=0.00s
[expr_001_lin_bi] run=0 seed=0 train=5.783e-30 interp=5.075e-30 extrap=1.537e-29 dur=0.00s
[expr_001_lin_bi] run=1 seed=1 train=5.783e-30 interp=5.075e-30 extrap=1.537e-29 dur=0.00s
[expr_001_lin_bi] run=2 seed=2 train=5.783e-30 interp=5.075e-30 extrap=1.537e-29 dur=0.00s
[expr_001_lin_bi] run=3 seed=3 train=5.783e-30 interp=5.075e-30 extrap=1.537e-29 dur=0.00s
[expr_001_lin_bi] run=4 seed=4 train=5.783e-30 interp=5.075e-30 extrap=1.537e-29 dur=0.00s
[expr_002_poly2_uni] run=0 seed=0 train=1.242e-30 interp=1.479e-30 extrap=6.939e-29 d

In [2]:
from __future__ import annotations

import math
from pathlib import Path

import pandas as pd
import sympy as sp

from config.benchmark_config import SINDY


def _safe_latex(expr_str: str) -> str | None:
    if not isinstance(expr_str, str) or not expr_str.strip():
        return None
    try:
        expr = sp.sympify(expr_str)
        return sp.latex(expr)
    except Exception:
        return None


def _format_pm(value: float, std: float, sig: int = 1) -> str:
    if value == 0.0:
        return r"(0\pm0)\times 10^{0}"

    exp = int(math.floor(math.log10(abs(value))))
    scale = 10 ** exp

    v = round(value / scale, sig)
    s = round(std / scale, sig)

    return rf"({v}\pm{s})\times 10^{{{exp}}}"


def _count_nodes(expr: sp.Expr) -> int:
    return sum(1 for _ in sp.preorder_traversal(expr))


def summarize(csv_path: str | Path, k: int = 5) -> None:
    df = pd.read_csv(csv_path)

    metrics = [
        "train_mse",
        "test_interp_mse",
        "test_extrap_mse",
    ]

    for gname, gdf in df.groupby("group"):
        print(f"\n{gname}")

        # Select k best by extrapolation error
        gdf = gdf.sort_values("test_extrap_mse").iloc[:k]

        # ---- error metrics ----
        for m in metrics:
            vals = gdf[m].astype(float).to_numpy()
            mean = float(vals.mean())
            std = float(vals.std(ddof=0))
            print(f"  {m}: {_format_pm(mean, std)}")

        # ---- symbolic complexity ----
        node_counts = []
        for s in gdf["found_expr"]:
            if not isinstance(s, str) or not s.strip():
                continue
            try:
                expr = sp.sympify(s)
                node_counts.append(_count_nodes(expr))
            except Exception:
                pass

        if node_counts:
            nc = pd.Series(node_counts, dtype=float)
            print(
                f"  expr_nodes: "
                f"{nc.mean():.1f} ± {nc.std(ddof=0):.1f}"
            )
        else:
            print("  expr_nodes: N/A")

        # ---- best train-fit expression ----
        best_row = gdf.sort_values("train_mse").iloc[0]
        latex_expr = _safe_latex(best_row["found_expr"])

        if latex_expr is not None:
            print("  best_train_expr_latex:")
            print(f"    ${latex_expr}$")
        else:
            print("  best_train_expr_latex: N/A")


if __name__ == "__main__":
    summarize(SINDY.results_path, k=5)


expr_000_lin_uni
  train_mse: (0\pm0)\times 10^{0}
  test_interp_mse: (0\pm0)\times 10^{0}
  test_extrap_mse: (0\pm0)\times 10^{0}
  expr_nodes: 5.0 ± 0.0
  best_train_expr_latex:
    $1.87 x_{1} + 2.01$

expr_001_lin_bi
  train_mse: (5.8\pm0.0)\times 10^{-30}
  test_interp_mse: (5.1\pm0.0)\times 10^{-30}
  test_extrap_mse: (1.5\pm0.0)\times 10^{-29}
  expr_nodes: 8.0 ± 0.0
  best_train_expr_latex:
    $1.56 x_{1} + 1.59 x_{2} - 2.91$

expr_002_poly2_uni
  train_mse: (1.2\pm0.0)\times 10^{-30}
  test_interp_mse: (1.5\pm0.0)\times 10^{-30}
  test_extrap_mse: (6.9\pm0.0)\times 10^{-29}
  expr_nodes: 10.0 ± 0.0
  best_train_expr_latex:
    $2.48 x_{1}^{2} + 1.92 x_{1} - 0.680000000000001$

expr_003_poly2_bi
  train_mse: (5.9\pm0.0)\times 10^{-29}
  test_interp_mse: (5.7\pm0.0)\times 10^{-29}
  test_extrap_mse: (4.9\pm0.0)\times 10^{-28}
  expr_nodes: 22.0 ± 0.0
  best_train_expr_latex:
    $0.550000000000001 x_{1}^{2} + 2.45 x_{1} x_{2} + 1.65 x_{1} + 2.95 x_{2}^{2} + 0.800000000000001 x